# Load and structure in samples and labels

In [1]:
import numpy as np
import mne
from pathlib import Path

# ---------------------------------------
# Basic config
# ---------------------------------------
root = Path("../../Datasets/EEG Motor Movement/original/files")
imagery_runs = {
    4: ("LR", "left_right"),
    8: ("LR", "left_right"),
    12: ("LR", "left_right"),
    6: ("FF", "fists_feet"),
    10: ("FF", "fists_feet"),
    14: ("FF", "fists_feet"),
}

# Label mapping
# 0 = left
# 1 = right
# 2 = both fists
# 3 = both feet

all_data = {}

# ---------------------------------------
# Loop over subjects
# ---------------------------------------
for subj in range(1, 110):   # 109 subjects
    subj_id = f"S{subj:03d}"
    subj_path = root / subj_id
    
    if not subj_path.exists():
        continue
    
    all_data[subj_id] = {}

    for run, (rtype, run_name) in imagery_runs.items():
        edf_path = subj_path / f"{subj_id}R{run:02d}.edf"
        if not edf_path.exists():
            continue

        raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
        raw.pick("eeg")

        # Clean channel names
        new_names = {ch: ch.strip(".").capitalize() for ch in raw.ch_names}
        raw.rename_channels(new_names)
        raw.set_montage("standard_1020", on_missing="ignore", verbose=False)

        events, event_id = mne.events_from_annotations(raw)

        # Only T1 and T2
        mi_event_id = {k: v for k, v in event_id.items() if k in ["T1", "T2"]}

        epochs = mne.Epochs(
            raw,
            events,
            event_id=mi_event_id,
            tmin=0.5,
            tmax=3.5,
            baseline=None,
            preload=True,
            verbose=False,
        )

        X = epochs.get_data()
        y_raw = epochs.events[:, -1]

        # Map labels according to run type
        y = []
        for label in y_raw:
            if rtype == "LR":
                if label == event_id["T1"]:
                    y.append(0)  # left
                else:
                    y.append(1)  # right
            else:
                if label == event_id["T1"]:
                    y.append(2)  # fists
                else:
                    y.append(3)  # feet

        y = np.array(y)

        all_data[subj_id][f"run_{run:02d}"] = {"X": X, "y": y}

print("✅ Finished loading PhysioNet 4-class imagery dataset.")

Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]

/var/folders/7c/12d1bdbs4cvbh7zksb_8wcgw0000gn/T/ipykernel_59890/124391646.py:43: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
/var/folders/7c/12d1bdbs4cvbh7zksb_8wcgw0000gn/T/ipykernel_59890/124391646.py:43: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
/var/folders/7c/12d1bdbs4cvbh7zksb_8wcgw0000gn/T/ipykernel_59890/124391646.py:43: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
/var/folders/7c/12d1bdbs4cvbh7zksb_8wcgw0000gn/T/ipykernel_59890/124391646.py:43: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
/var/folders/7c/12d1bdbs4cvbh7zksb_8wcgw0000gn/T/ipykernel_59890/124

Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]

# Filters and Feature extractions

In [2]:
from scipy.signal import butter, sosfiltfilt
from copy import deepcopy

def band_filter(data, fs=160.0, band=(4, 40), order=5):
    nyq = fs / 2.0
    low, high = band[0] / nyq, band[1] / nyq
    sos = butter(order, [low, high], btype="band", output="sos")
    return sosfiltfilt(sos, data, axis=-1)

filtered_data = deepcopy(all_data)

for subj_id, runs in all_data.items():
    for run_name, run_data in runs.items():
        X = run_data["X"]
        y = run_data["y"]

        X_filt = band_filter(X, fs=160.0, band=(4, 40), order=5)

        filtered_data[subj_id][run_name]["X"] = X_filt
        filtered_data[subj_id][run_name]["y"] = y

print("✅ All epochs filtered (4–40 Hz).")

✅ All epochs filtered (4–40 Hz).


In [3]:
import pandas as pd
from numpy.fft import fft
from tqdm import tqdm

def compute_time_cov(matrix):
    return np.cov(matrix)

def compute_freq_cov(matrix):
    fft_vals = np.abs(fft(matrix, axis=0))
    return np.cov(fft_vals)

def flatten_covariance(cov, prefix):
    idx = np.triu_indices_from(cov)
    vals = cov[idx]
    names = [f"{prefix}{i}_{j}" for i, j in zip(idx[0], idx[1])]
    return vals, names

all_features = []

for subj_id, runs in tqdm(filtered_data.items(), desc="Subjects"):
    for run_name, run_data in runs.items():
        X = run_data["X"]
        y = run_data["y"]

        for trial_idx, trial in enumerate(X):

            cov_t = compute_time_cov(trial)
            cov_t_vals, cov_t_names = flatten_covariance(cov_t, prefix="time_")

            cov_f = compute_freq_cov(trial)
            cov_f_vals, cov_f_names = flatten_covariance(cov_f, prefix="freq_")

            feature_vals = np.concatenate([cov_t_vals, cov_f_vals])
            feature_names = cov_t_names + cov_f_names

            all_features.append({
                "subject": subj_id,
                "run": run_name,
                "label": int(y[trial_idx]),
                **{feature_names[i]: feature_vals[i] for i in range(len(feature_vals))}
            })

df_features = pd.DataFrame(all_features)

print("Shape:", df_features.shape)
df_features.head()

Subjects: 100%|██████████| 109/109 [01:00<00:00,  1.81it/s]


Shape: (9837, 4163)


,subject,run,label,time_0_0,time_0_1,time_0_2,time_0_3,time_0_4,time_0_5,time_0_6,...,freq_60_60,freq_60_61,freq_60_62,freq_60_63,freq_61_61,freq_61_62,freq_61_63,freq_62_62,freq_62_63,freq_63_63
0,S001,run_04,1,6.846964e-10,6.518054e-10,6.077409e-10,5.465176e-10,5.075677e-10,4.255543e-10,3.271100e-10,...,5.085506e-09,3.796836e-09,4.181220e-09,7.742553e-09,1.057665e-08,9.293368e-09,1.313325e-08,2.091977e-08,1.775934e-08,3.048525e-08
1,S001,run_04,0,5.120350e-10,4.920864e-10,4.669942e-10,4.341281e-10,3.996609e-10,3.433887e-10,2.724267e-10,...,6.832625e-09,3.881552e-09,3.022016e-09,8.891510e-09,8.040003e-09,4.997622e-09,1.092952e-08,1.304289e-08,1.158778e-08,2.975252e-08
2,S001,run_04,0,6.873191e-10,6.344463e-10,5.969032e-10,5.181424e-10,4.780643e-10,3.873522e-10,2.556222e-10,...,5.283051e-09,3.711268e-09,2.883768e-09,8.311813e-09,1.120947e-08,6.936272e-09,1.605568e-08,1.503982e-08,1.435841e-08,3.845416e-08
3,S001,run_04,1,4.020172e-10,3.786990e-10,3.662724e-10,3.408140e-10,3.082553e-10,2.473830e-10,1.663981e-10,...,5.640294e-09,3.224054e-09,3.669789e-09,7.206327e-09,6.758765e-09,6.356468e-09,9.318399e-09,1.628608e-08,1.519752e-08,2.697751e-08
4,S001,run_04,1,4.748152e-10,4.441843e-10,4.237454e-10,3.781393e-10,3.492971e-10,2.933665e-10,2.176888e-10,...,4.596763e-09,2.022426e-09,1.697734e-09,5.271222e-09,5.555512e-09,2.822636e-09,6.543944e-09,1.127478e-08,5.707967e-09,2.079408e-08


In [9]:
df_features["label"].value_counts()

label
0    2479
2    2465
3    2455
1    2438
Name: count, dtype: int64

In [10]:
df_features["subject"].value_counts()

subject
S088    114
S092    114
S001     90
S069     90
S078     90
       ... 
S031     90
S030     90
S109     90
S104     87
S100     72
Name: count, Length: 109, dtype: int64

# Save

In [12]:
df_features.to_csv("../../Datasets/EEG Motor Movement/processed/EEG_MM_features.csv", index=False)